# Browser runtime smoke check

This compact technical notebook is used by automated browser checks. It confirms that JupyterLite can load the shared helper, public synthetic data, GeoPandas, mapclassify, and statsmodels. It is not a lesson or assessment.

In [ ]:
from __future__ import annotations

import os
import sys
import types
from urllib.request import urlopen

if sys.platform == "emscripten":
    import piplite
    await piplite.install("pyodide-http")
    import pyodide_http
    pyodide_http.patch_all()

RAW_CODE_ROOT = "https://raw.githubusercontent.com/muzammilafroz/isb-spia-data-test-learning-lab/v1.0.0/learning_lab"
if sys.platform == "emscripten":
    from js import window
    if window.location.hostname in {"127.0.0.1", "localhost"}:
        RAW_CODE_ROOT = f"{window.location.origin}/learning_lab"
        os.environ["LEARNING_LAB_DATA_BASE"] = f"{window.location.origin}/data/teaching"

def load_public_module(module_name):
    """Import locally, or fetch the small public helper when running in Colab/Lite."""
    try:
        return __import__(f"learning_lab.{module_name}", fromlist=[module_name])
    except ModuleNotFoundError:
        location = f"{RAW_CODE_ROOT}/{module_name}.py"
        source = urlopen(location).read().decode("utf-8")
        module = types.ModuleType(f"learning_lab.{module_name}")
        exec(compile(source, location, "exec"), module.__dict__)
        return module

lab_io = load_public_module("io")
get_data_url = lab_io.get_data_url
read_teaching_csv = lab_io.read_teaching_csv
read_teaching_geojson = lab_io.read_teaching_geojson

print("Runtime:", sys.platform)
print("Data reference:", os.getenv("LEARNING_LAB_DATA_REF", "v1.0.0"))

In [ ]:
import pandas as pd
import geopandas as gpd
if sys.platform == "emscripten":
    await piplite.install("mapclassify")
import mapclassify
import statsmodels.formula.api as smf

districts = read_teaching_geojson()
production = read_teaching_csv("district_production.csv")
analysis = read_teaching_csv("synthetic_analysis_clean.csv", dtype={"hhid": "string"})

mapped = districts.merge(production, on="d", validate="one_to_one")
classifier = mapclassify.FisherJenks(mapped["production"], k=5)
model = smf.ols("positive_rate ~ wealth_index", data=analysis).fit(cov_type="HC1")

assert len(mapped) == 25
assert len(classifier.bins) == 5
assert model.params["wealth_index"] < 0
print("JUPYTERLITE_SMOKE_OK", len(mapped), len(classifier.bins), round(model.params["wealth_index"], 6))